In [12]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "src").exists() else cwd.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.data import load_sales
from src.models import SeasonalDateRegressor, SklearnRegressorConfig, SklearnRegressorWrapper
from src.pipelines.train import split_by_time
from src.evaluation import regression_metrics
from src.features import add_time_features, add_lag_features, add_rolling_features
from src.utils import prepare_submission

data_root = repo_root / "data" / "datathon-2026-round-1"

In [7]:
df = load_sales(data_root=data_root, parse_dates=True)
train_df, valid_df = split_by_time(df, date_col="Date", valid_fraction=0.2)
model = SeasonalDateRegressor()

X_train = train_df[["Date"]]
y_train = train_df["Revenue"]

model.fit(X_train, y_train)

y_pred = model.predict(valid_df[["Date"]])
y_true = valid_df["Revenue"]
metrics = regression_metrics(y_true, y_pred)
metrics

{'mae': 1644555.194360604,
 'rmse': 1965403.7315175715,
 'mape': 69.72676924136569,
 'smape': 47.24199506693533,
 'r2': -0.386503302378604}

In [17]:
# Train two models on the same feature space:
# 1) Revenue model
# 2) Ratio model for revenue_to_cogs = Revenue / COGS

df = load_sales(data_root=data_root, parse_dates=True)
df = add_time_features(df, date_col="Date")
df = add_lag_features(df, target_col="Revenue", lags=[1, 7, 14], date_col="Date")
df = add_rolling_features(df, target_col="Revenue", windows=[7, 14], date_col="Date")
df = df.dropna().reset_index(drop=True)

train_df, valid_df = split_by_time(df, date_col="Date", valid_fraction=0.2)

feature_cols = [col for col in df.columns if col not in ["Date", "Revenue", "COGS"]]
X_train = train_df[feature_cols]
X_valid = valid_df[feature_cols]

y_train_revenue = train_df["Revenue"]
y_valid_revenue = valid_df["Revenue"]

# Ratio target (guard against zero COGS in training)
eps = 1e-6
train_cogs_safe = train_df["COGS"].clip(lower=eps)
y_train_ratio = y_train_revenue / train_cogs_safe

model_config = SklearnRegressorConfig(model_type="random_forest")

# Validation-stage models
model_rev = SklearnRegressorWrapper(model_config)
model_rev.fit(X_train, y_train_revenue)
y_pred_revenue = pd.Series(model_rev.predict(X_valid), index=valid_df.index, dtype="float64")

model_ratio = SklearnRegressorWrapper(model_config)
model_ratio.fit(X_train, y_train_ratio)
y_pred_ratio = pd.Series(model_ratio.predict(X_valid), index=valid_df.index, dtype="float64")

# Reconstruct COGS from Revenue and ratio
y_pred_ratio = y_pred_ratio.clip(lower=eps)
y_pred_cogs = y_pred_revenue / y_pred_ratio

# Evaluate reconstructed targets on validation
metrics_revenue = regression_metrics(y_valid_revenue, y_pred_revenue)
metrics_cogs_recon = regression_metrics(valid_df["COGS"], y_pred_cogs)

print("Revenue metrics:", metrics_revenue)
print("Reconstructed COGS metrics:", metrics_cogs_recon)

# Retrain on all available rows for final forecasting
X_all = df[feature_cols]
y_all_revenue = df["Revenue"]
y_all_ratio = y_all_revenue / df["COGS"].clip(lower=eps)

model_rev = SklearnRegressorWrapper(model_config)
model_rev.fit(X_all, y_all_revenue)

model_ratio = SklearnRegressorWrapper(model_config)
model_ratio.fit(X_all, y_all_ratio)

# Safety bounds for reconstructed COGS in submission
cogs_lower_bound = float(df["COGS"].quantile(0.005))
cogs_upper_bound = float(df["COGS"].quantile(0.995))
print(f"Using COGS clip bounds: [{cogs_lower_bound:.2f}, {cogs_upper_bound:.2f}]")

Revenue metrics: {'mae': 560298.1185008507, 'rmse': 808744.6214439586, 'mape': 23.395869216465027, 'smape': 20.314012107137707, 'r2': 0.7652289065545682}
Reconstructed COGS metrics: {'mae': 513989.96614526986, 'rmse': 752954.0034694931, 'mape': 22.837494301836287, 'smape': 20.733472281211014, 'r2': 0.7301287102355446}
Using COGS clip bounds: [629257.63, 12928811.67]


In [18]:
# Build final submission with recursive lag/rolling features from predicted revenue
sub_start = "2023-01-01"
sub_end = "2024-07-01"
sub_dates = pd.date_range(sub_start, sub_end, freq="D")

lags = [1, 7, 14]
windows = [7, 14]
pred_revenue_history: list[float] = []
pred_ratio_history: list[float] = []

for current_date in sub_dates:
    row = pd.DataFrame({"Date": [current_date]})
    row = add_time_features(row, date_col="Date")

    # Create lag features from previous predicted revenues
    for lag in lags:
        if len(pred_revenue_history) >= lag:
            row[f"Revenue_lag_{lag}"] = pred_revenue_history[-lag]
        else:
            row[f"Revenue_lag_{lag}"] = 0.0

    # Create rolling features using previous predicted revenues
    for window in windows:
        if len(pred_revenue_history) >= window:
            values = pd.Series(pred_revenue_history[-window:], dtype="float64")
            row[f"Revenue_roll_mean_{window}"] = values.mean()
            row[f"Revenue_roll_std_{window}"] = values.std()
        else:
            row[f"Revenue_roll_mean_{window}"] = 0.0
            row[f"Revenue_roll_std_{window}"] = 0.0

    X_row = row.reindex(columns=feature_cols, fill_value=0.0)

    pred_rev = float(model_rev.predict(X_row)[0])
    pred_ratio = max(float(model_ratio.predict(X_row)[0]), eps)

    pred_revenue_history.append(pred_rev)
    pred_ratio_history.append(pred_ratio)

sub_pred_revenue = pd.Series(pred_revenue_history, dtype="float64")
sub_pred_ratio = pd.Series(pred_ratio_history, dtype="float64")

# Reconstruct and clip COGS for numerical safety
sub_pred_cogs = (sub_pred_revenue / sub_pred_ratio).clip(
    lower=cogs_lower_bound,
    upper=cogs_upper_bound,
 )

submission = prepare_submission(
    revenue=sub_pred_revenue.values,
    cogs=sub_pred_cogs.values,
    start_date=sub_start,
    end_date=sub_end,
 )

submission.head(), submission.tail()

(         Date       Revenue          COGS
 0  2023-01-01  9.120988e+05  8.887809e+05
 1  2023-01-02  1.254100e+06  1.086299e+06
 2  2023-01-03  1.430334e+06  1.204956e+06
 3  2023-01-04  1.386725e+06  1.162492e+06
 4  2023-01-05  1.451797e+06  1.210270e+06,
            Date       Revenue          COGS
 543  2024-06-27  7.993873e+06  8.019908e+06
 544  2024-06-28  7.795836e+06  7.794025e+06
 545  2024-06-29  7.652199e+06  7.648957e+06
 546  2024-06-30  7.998704e+06  7.983701e+06
 547  2024-07-01  7.580417e+06  7.570653e+06)

In [21]:
import os
os.makedirs(repo_root / "submissions", exist_ok=True)
submission.to_csv(repo_root / "submissions" / "baseline_submission.csv", index=False)